# M2 · Physical profiling — where does GPU time go?

**Learning objective:** collect a real kernel trace of the Qwen-Image workload on your W7900 with `rocprofv3`, attribute GPU busy time to GEMM / attention / norm / elementwise classes, and separate *GPU busy time* from *end-to-end latency*. The frozen campaign's 61.3% GEMM share is a hypothesis for your host.

Reference lesson: [M02 lesson](../course/M02_profiling/LESSON.md) · [中文](./02_profile_and_bottlenecks.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="02")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · Pipeline decomposition from run records
First the cheap layer: split warm end-to-end latency into KSampler (DiT denoise), VAE decode, text-encode and derived residual, using node timings from your (or the reference) B0 records.

In [ ]:
def own_pipeline_layer():
    out = session.run_profile(layers="pipeline")
    return out

pipe_dir = gpu_cell(own_pipeline_layer)
if pipe_dir is not None:
    pb = json.loads((pipe_dir / "pipeline_breakdown.json").read_text())
    _stamp(MEASURED_TODAY, f"pipeline layer from YOUR records: {pipe_dir.name}")
else:
    pb = ns.read_json("evidence/results/p2-profile-profile/pipeline_breakdown.json")
    _stamp(HISTORICAL, "frozen campaign pipeline breakdown")
for k in ("n_warm", "T_total_warm_s", "T_dit_denoise_s", "T_vae_decode_s",
          "T_text_encoder_cached_s", "T_other_warm_s", "step_median_s"):
    print(f"  {k:28s} {pb.get(k)}")
print("  method note:", pb.get("method", {}).get(
    "T_other_warm_s", pb.get("note", "derived by subtraction")))

## 2 · The kernel layer (rocprofv3)
The expensive layer traces the **real ComfyUI server process** under a bounded `rocprofv3` window (DiT phase and text-encoder phase). Interception adds overhead, so **traced wall times are never performance claims** — composition only. In class you usually *consume* the qualification-campaign trace; `RUN_KERNEL_TRACE = True` (GPU modes) launches your own bounded window (~10 min).

In [ ]:
RUN_KERNEL_TRACE = False  # opt-in: ~10 min of physical GPU tracing

FROZEN_TRACE = ROOT / "evidence/results/p2-profile-profile/rocprof-dit/analysis.json"

_USABLE_TRACE = ("CURRENT_SESSION_PHYSICAL", "SAME_STUDENT_PRIOR_PHYSICAL",
                 "SHARED_CAMPAIGN_PHYSICAL", "COMMITTED_PHYSICAL_REFERENCE")

def load_kernel_analysis():
    """Explicit, provenance-scoped trace selection (no latest-file-wins):
    only traces this session produced, an earlier verified session of this
    workspace produced, or a declared campaign / committed reference."""
    cands = sorted((ROOT / "runs").glob("*-profile/rocprof-dit/analysis.json"))
    for p in reversed(cands):
        rid = p.parent.parent.name
        try:
            cat = prov.classify_run(rid, session_id=session.session_id)["category"]
        except prov.ProvenanceError:
            continue
        if cat in _USABLE_TRACE:
            print(f"trace source: run_id={rid}")
            return json.loads(p.read_text()), _origin_stamp(cat), rid
    if FROZEN_TRACE.is_file():
        return json.loads(FROZEN_TRACE.read_text()), HISTORICAL, "evidence/results/p2-profile-profile"
    return None, None, None

loaded = gpu_cell(load_kernel_analysis)
ka = loaded[0] if loaded else None
ka_origin = loaded[1] if loaded else None
ka_rid = loaded[2] if loaded else None
if ka is None and RUN_KERNEL_TRACE and session.mode != "REFERENCE":
    session.run_profile(layers="pipeline,kernel")
    loaded = load_kernel_analysis()
    ka, ka_origin, ka_rid = (loaded[0], loaded[1], loaded[2]) if loaded else (None, None, None)
if ka is not None:
    _stamp(ka_origin, f"kernel trace analysis ({ka_rid}; traced run; composition only)")
    print("mode:", ka.get("mode"), "| kernels:", ka.get("n_kernels"),
          "| gpu_busy_s:", ka.get("gpu_busy_s"), f"({ka.get('gpu_busy_pct')}%)")
    by = ka.get("by_class", {})
    total = sum(v["total_s"] for v in by.values()) or 1.0
    for name, v in sorted(by.items(), key=lambda kv: -kv[1]["total_s"]):
        print(f"  {name:12s} {v['total_s']:8.2f}s  {v['total_s']/total*100:5.1f}%  "
              f"n={v.get('n', v.get('count', '?'))}")
else:
    print("No kernel trace available on this host")

## 3 · From composition to an Amdahl ceiling
If GEMM is fraction `g` of GPU busy time, even an infinitely fast GEMM caps the kernel-layer speedup at 1/(1−g). Compute the ceiling from **your** trace — and remember the ceiling applies to GPU busy time, not end-to-end latency (queueing, TE, VAE, host overhead sit outside it).

In [ ]:
if ka is not None:
    by = ka.get("by_class", {})
    gemm = by.get("gemm", {}).get("total_s", 0.0)
    total_busy = sum(v["total_s"] for v in by.values()) or 1.0
    g = gemm / total_busy
    ceiling = 1.0 / (1.0 - g) if g < 1 else float("inf")
    _stamp(ESTIMATE,
           f"Amdahl kernel-layer ceiling {ceiling:.2f}x from YOUR measured "
           f"g={g:.3f} (derived, not measured)")
    print(f"gemm share g = {g*100:.1f}%  ->  ceiling {ceiling:.2f}x")
else:
    g, ceiling = 0.613, 1.0 / (1 - 0.613)
    _stamp(HISTORICAL, f"frozen campaign g=61.3% -> ceiling {ceiling:.2f}x")

## 4 · Your turn
1. Why can the profiled run be slower end-to-end while the kernel composition stays valid?
2. Your end-to-end warm median (M1) is much larger than total GPU busy time — list three concrete contributors to the gap.
3. Which class would you attack next, and what is its own Amdahl ceiling if perfectly eliminated?

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment B0 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.